# Module 10 Part 2: SQL for Your Project - Your Own Neon or DuckDB

**BAN 6003: Data Management and Analytics Integration**

This notebook is a **guide, not a graded lab**. It shows the two ways to meet the SQL requirement of the course project. Read Section 1, choose one path, and work through that part. You may try both.

> Project rule: your project notebook must include at least one SQL query on your own project data, using **Path A** or **Path B**.

## 1. Choose Your Path

| | Path A: your own Neon database | Path B: DuckDB |
|---|---|---|
| What it is | A free Postgres database in the cloud that **you** own | A small database engine that runs inside Python |
| Account needed | Yes, a free Neon account | No |
| Where your data lives | You upload your CSV files or DataFrames to your database | Your DataFrames and CSV files stay where they are |
| Connection string | Yes, saved in `.env` as `PROJECT_DATABASE_URL` | No |
| SQL you write | Postgres SQL, as in Modules 9-10 | Almost the same SQL (see Section 4) |
| Good for | Practicing a real database workflow: create, upload, query | The fastest way to add SQL checks to your project |

Both paths count the same for the project. Choose Path A if you want hands-on practice with your own database. Choose Path B if you want the simplest setup.

**How to use this notebook in your project.** Your project data is in your **project** repository, not in this one. Cells whose first line says `# COPY` are written so you can copy them into your project notebook without changes. Practice here first with a small practice file, then copy the cells into your project notebook and run them there.

## 2. Setup

This cell loads the packages and creates a small practice CSV file, `data/processed/practice_planes.csv`, with one row per aircraft (3,322 rows). The rest of this notebook uses it as stand-in project data.

In [ ]:
# Codespaces: these packages are installed from requirements.txt.
# Local only: if an import fails, uncomment and run this line:
# %pip install pandas sqlalchemy psycopg2-binary "psycopg[binary]" python-dotenv duckdb nycflights13 "setuptools<81"
# (setuptools<81 is needed because nycflights13 uses pkg_resources, which newer setuptools removed.)

from pathlib import Path

import pandas as pd
from nycflights13 import planes

# Find the top folder of the repository (the folder that contains README.md).
repo_root = Path.cwd()
if not (repo_root / "README.md").exists() and (repo_root.parent / "README.md").exists():
    repo_root = repo_root.parent

practice_csv = repo_root / "data" / "processed" / "practice_planes.csv"
practice_csv.parent.mkdir(parents=True, exist_ok=True)
planes.to_csv(practice_csv, index=False)
print("Practice file:", practice_csv.relative_to(repo_root), "| rows:", len(planes))


## 3. Path A: Your Own Neon Database

In Modules 9-10 you **read** from the course database. Here you create your **own** database, so you can also **write**: you upload your project tables and then query them with SQL.

Path A has four steps:

1. Create a free Neon project and copy its connection string (Section 3.1).
2. Save the string in `.env` as `PROJECT_DATABASE_URL` and connect (Section 3.2).
3. Upload your CSV files or DataFrames (Section 3.3).
4. Query your tables with SQL and save the results (Section 3.4).

If you chose Path B, you can skip to Section 4. The Path A cells below still run without errors: they print a message and skip the database steps.

### 3.1 Create your Neon project and copy the connection string

1. Go to [neon.com](https://neon.com) and sign up for a free account. Signing up with your GitHub account is the easiest option.
2. Right after you sign up, Neon asks you to **create your first project**. Give it a clear name such as `ban6003-project`. For everything else, the default choices are fine.
3. Neon then shows a page that says your new project was created, with a section called **Let your agent take it from here** and a quickstart prompt. You do **not** need this. Click **Take me to project**.
4. If you see a list of projects, click the project you just created.
5. Click the **Connect** button at the top left. A window opens, and your connection string is at the bottom of it.
6. In that window the password shows as `****`. This is normal: Neon hides it on screen. Click **Copy snippet**. The copied text contains the full connection string with the real password.
7. Paste it into `.env` in Section 3.2. The string starts with `postgresql://`. If the copied text has anything in front of `postgresql://`, keep only the part that starts with `postgresql://`.

**Your connection string contains your password and gives full access to your database.** Keep it only in `.env`. Never paste it into a notebook, a GitHub file, a Canvas post, or a chat. If it is ever exposed, reset the password in the Neon Console (Roles) and update `.env`.

### What you have now

You now own a complete Postgres database in the cloud. Unlike the read-only course database, you can **write** to it:

- **From Python** (this notebook and your project notebook): upload tables (Section 3.3) and query them with the same SQL you used in Modules 9-10 (Section 3.4).
- **In the Neon Console** (the website): open **Tables** to browse the tables you uploaded, and open the **SQL Editor** to run SQL in your browser. The SQL Editor is a quick way to test a query before you put it into your notebook.

Good to know about the free plan:

- Each project can store up to 1 GB, which is much more than the course project data needs.
- The database goes to sleep after 5 minutes without queries. The first query after a break can take a few seconds while it wakes up. This is normal.

### 3.2 Save the connection string and connect

Run the next cell. The first time, it adds an empty line `PROJECT_DATABASE_URL=` to your `.env` file (the file is created if it does not exist) and tells you what to do. Open `.env`, paste your **own** connection string right after `PROJECT_DATABASE_URL=` (no spaces, no quotes), save the file, and run the cell again.

Use the name `PROJECT_DATABASE_URL` only for your own database. `DATABASE_URL` stays the read-only course database.

The cell also creates the folder `outputs/sql_results`. Section 3.4 explains why.

**In your project:** copy this cell into your project notebook. Your project repository has its own `.env` file (Git ignores it there too), so paste your connection string there as well.

In [ ]:
# COPY - Path A, cell 1 of 3: connect to your own Neon database
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Find the top folder of the repository (the folder that contains README.md).
repo_root = Path.cwd()
if not (repo_root / "README.md").exists() and (repo_root.parent / "README.md").exists():
    repo_root = repo_root.parent

# Add an empty PROJECT_DATABASE_URL line to .env the first time. Git ignores .env.
env_path = repo_root / ".env"
env_text = env_path.read_text() if env_path.exists() else ""
if "PROJECT_DATABASE_URL" not in env_text:
    if env_text and not env_text.endswith("\n"):
        env_text += "\n"
    env_text += ("# Your own Neon database (Path A). Paste your connection string after the = sign.\n"
                 "PROJECT_DATABASE_URL=\n")
    env_path.write_text(env_text)
    print("Added an empty PROJECT_DATABASE_URL line to .env")

load_dotenv(env_path, override=True)
project_url = os.getenv("PROJECT_DATABASE_URL")

# Saved query results go here, so the notebook also runs without your .env (see Section 3.4).
SQL_RESULTS_DIR = repo_root / "outputs" / "sql_results"
SQL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if project_url:
    project_engine = create_engine(project_url)
    with project_engine.connect() as conn:
        conn.execute(text("SELECT 1;"))
    print("Connected to your own database. For safety, we do not print the connection string.")
else:
    project_engine = None
    print("PROJECT_DATABASE_URL is empty, so the database steps will be skipped.")
    print("Path A: open .env, paste your own Neon connection string after PROJECT_DATABASE_URL=, save, and run this cell again.")


### 3.3 Upload your data

`upload_table()` copies one table into your database:

- give it a **CSV file path** to upload a raw data file, or
- give it a **DataFrame** to upload a table you cleaned or built in Pandas, for example your ABT.

It writes the table with `to_sql(..., if_exists="replace")`, so running it again replaces the old copy instead of adding duplicate rows. Then it compares the row count in the database with the row count in Python. The two numbers should be equal.

Table names: use lowercase letters, numbers, and underscores, for example `movie_abt` or `order_items`. Then you never need quotes around the name in SQL.

**Column names with capital letters need double quotes in Postgres.** The upload keeps your column names exactly as they are in Pandas. Postgres turns unquoted names into lowercase, so a column such as `movieId` must be written as `"movieId"` in your SQL, for example `COUNT(DISTINCT "movieId")`. Without the quotes, Postgres looks for `movieid` and reports that the column does not exist. (DuckDB in Path B does not have this problem.)

In [ ]:
# COPY - Path A, cell 2 of 3: upload a CSV file or a DataFrame to your database
def upload_table(source, table_name):
    """Upload a CSV file path or a DataFrame to your own database and check the row count."""
    if project_engine is None:
        print(f"Skipped uploading {table_name}: no PROJECT_DATABASE_URL in .env.")
        return
    df = source if isinstance(source, pd.DataFrame) else pd.read_csv(source, low_memory=False)
    df.to_sql(table_name, project_engine, if_exists="replace", index=False, chunksize=1000, method="multi")
    with project_engine.connect() as conn:
        db_rows = conn.execute(text(f"SELECT COUNT(*) FROM {table_name};")).scalar()
    status = "OK" if db_rows == len(df) else "CHECK: the row counts differ"
    print(f"{table_name}: {len(df):,} rows in Python, {db_rows:,} rows in the database. {status}")


# Fill in your own files. One line per table: upload_table(path_or_dataframe, "table_name")
# In your project, use your data files, for example:
#   upload_table(repo_root / "data" / "ratings.csv", "ratings")
#   upload_table(abt, "movie_abt")      # a DataFrame you built in Pandas
upload_table(practice_csv, "practice_planes")


You only need to upload a table again when its data changes. If your ABT changes, run `upload_table(abt, "your_abt_name")` again to replace the copy in the database.

In your project, `practice_csv` does not exist: replace the last line with your own `upload_table(...)` lines.

### 3.4 Query your tables, and keep the notebook runnable for the instructor

Your `.env` file is not on GitHub, so the instructor cannot connect to your database. To keep your project notebook runnable, `run_project_sql()` works in two modes:

- **With** `PROJECT_DATABASE_URL`: it runs the query on your database and saves the result as a CSV file in `outputs/sql_results`.
- **Without** it (for example, when the instructor runs your notebook): it shows the saved result instead and says so.

So: run every query once yourself, then **commit and push the `outputs/sql_results` files** together with your notebook. Never commit `.env`.

In [ ]:
# COPY - Path A, cell 3 of 3: run SQL on your database and save the result
def run_project_sql(query, result_name):
    """Run SQL on your own database and save the result. Without a connection, show the saved result."""
    result_path = SQL_RESULTS_DIR / f"{result_name}.csv"
    if project_engine is not None:
        result = pd.read_sql(text(query), project_engine)
        result.to_csv(result_path, index=False)
        print(f"Ran on your database. Saved the result to {result_path.relative_to(repo_root)}")
        return result
    if result_path.exists():
        print(f"No database connection here, so this is the saved result from {result_path.relative_to(repo_root)}")
        return pd.read_csv(result_path)
    print(f"Skipped {result_name}: no PROJECT_DATABASE_URL and no saved result yet.")
    return pd.DataFrame()


Example: a key check and a grouped summary on the practice table. In your project, replace the table and column names with yours, and give each query its own `result_name`.

In [ ]:
# Key check: does every row have its own tailnum?
key_check = run_project_sql("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT tailnum) AS distinct_tailnum
FROM practice_planes;
""", "practice_key_check")
key_check


In [ ]:
# Grouped summary: the five most common manufacturers
manufacturer_summary = run_project_sql("""
SELECT
    manufacturer,
    COUNT(*) AS plane_count,
    ROUND(AVG(seats)::numeric, 1)::float AS avg_seats
FROM practice_planes
GROUP BY manufacturer
ORDER BY plane_count DESC
LIMIT 5;
""", "practice_manufacturer_summary")
manufacturer_summary


If you connected, the key check shows 3,322 rows and 3,322 distinct tail numbers: one row per aircraft. If you did not connect and have no saved results, these cells print a skip message. That is fine if you chose Path B.

**Path A checklist for your project notebook:**

1. Copy the three `# COPY` cells into your project notebook.
2. Paste your own connection string into the project repository's `.env` as `PROJECT_DATABASE_URL`.
3. Replace the practice upload line with your own `upload_table(...)` lines and run them.
4. Write at least one SQL query with `run_project_sql(query, "a_clear_name")` that supports your integration or validation work.
5. Under each query, write one or two sentences on what the result tells you.
6. Commit and push the notebook and the files in `outputs/sql_results`. Check that `.env` is **not** in the commit.

## 4. Path B: DuckDB on Your DataFrames and CSV Files

In your course project, you work with **your own data**: Pandas DataFrames in your notebook and CSV files in your repository. Your project data is not in the course database.

**DuckDB** is a small database engine that runs inside Python, on your own computer or Codespace. It needs no account, no server, and no connection string. DuckDB can run SQL directly on:

- a Pandas DataFrame, by using the DataFrame's variable name as the table name, for example `FROM abt`;
- a CSV file, by using `read_csv('path/to/file.csv')` as the table name.

The SQL is almost the same as the SQL you used on Neon in Modules 9-10: `SELECT`, `WHERE`, `GROUP BY`, `HAVING`, `JOIN`, and CTEs with `WITH` all work. One small difference: in DuckDB, `ROUND(AVG(column), 2)` already gives normal numbers, so you do not need `::numeric` and `::float`.

The pattern is:

```python
duckdb.sql("SELECT ... FROM my_dataframe ...").df()
```

`.df()` turns the result into a Pandas DataFrame. Because DuckDB runs inside the notebook, the instructor can rerun your queries directly, so you do not need to save the results.

Path B does not use any database connection. If you skipped Path A, start here.

In [ ]:
# COPY - Path B: DuckDB setup
# Codespaces: duckdb is installed from requirements.txt.
# Local only: if the import below fails, uncomment and run this line:
# %pip install -q duckdb

import duckdb
import pandas as pd


def run_duckdb(query):
    """Run a DuckDB query when it has been filled in; keep blank practice cells runnable."""
    if not query or not query.strip():
        print("This practice cell is waiting for your SQL. Add your query inside the triple quotes, then run it again.")
        return pd.DataFrame()
    return duckdb.sql(query).df()


print("DuckDB version:", duckdb.__version__)


### 4.1 Practice data: a carrier-month table

The examples use a carrier-month summary of the flights data: one row per carrier per month. In Module 10, Section 7, you built the same table with SQL on Neon. Here we build it with Pandas, so this part needs no database. In your project, your own DataFrames (for example your ABT) take the place of this table.

In [ ]:
from nycflights13 import airlines, flights

carrier_month_summary = (
    flights
    .groupby(["carrier", "month"])
    .agg(
        flight_count=("flight", "size"),
        avg_dep_delay=("dep_delay", "mean"),
        avg_arr_delay=("arr_delay", "mean"),
        avg_distance=("distance", "mean"),
    )
    .round(2)
    .reset_index()
)

print("Rows:", len(carrier_month_summary))
carrier_month_summary.head()


### Example A: Query a DataFrame and run a two-way check

DuckDB rolls `carrier_month_summary` up to one row per carrier and keeps carriers with at least 10,000 flights.

Then we run a **two-way check**: Pandas answers the same question from the raw `flights` table, and we compare the two answers. If the carriers and flight counts match, we can trust the DuckDB query. In your project, the same idea works for any SQL step: compare it with your Pandas result.

In [ ]:
# DuckDB: the table name is the DataFrame name, carrier_month_summary
duck_carrier = run_duckdb('''
SELECT
    carrier,
    SUM(flight_count)::BIGINT AS total_flights,  -- ::BIGINT shows whole numbers
    COUNT(*) AS months_flown
FROM carrier_month_summary
GROUP BY carrier
HAVING SUM(flight_count) >= 10000
ORDER BY total_flights DESC;
''')

# Pandas: the same question, answered from the raw flights table
pandas_carrier = (
    flights.groupby("carrier").size().reset_index(name="total_flights")
    .query("total_flights >= 10000")
)

# Two-way check: same number of rows, same carriers, same flight counts
check = duck_carrier.merge(pandas_carrier, on="carrier", how="outer", suffixes=("_duckdb", "_pandas"))
print("Rows from DuckDB:", len(duck_carrier), "| Rows from Pandas:", len(pandas_carrier))
print("Same carriers and flight counts:", (check["total_flights_duckdb"] == check["total_flights_pandas"]).all())
duck_carrier


### Example B: Query a CSV file directly

In the project, much of your data is in CSV files in your repository. DuckDB can read a CSV file in the `FROM` clause, so you do not need to load it with `pd.read_csv` first.

First, we save `carrier_month_summary` as a small practice CSV file in `data/processed`. Then we query the file with `read_csv(...)`. `as_posix()` writes the path with forward slashes, which DuckDB accepts on every operating system.

In [ ]:
# Save a small practice CSV file (185 rows)
carrier_month_csv = repo_root / "data" / "processed" / "carrier_month_summary_practice.csv"
carrier_month_summary.to_csv(carrier_month_csv, index=False)
print("Querying the file:", carrier_month_csv.relative_to(repo_root))

# DuckDB: read_csv(...) with the file path is the table
month_from_file = run_duckdb(f'''
SELECT
    month,
    COUNT(*) AS carriers,
    SUM(flight_count)::BIGINT AS total_flights,  -- ::BIGINT shows whole numbers
    ROUND(AVG(avg_arr_delay), 2) AS avg_of_carrier_arr_delay
FROM read_csv('{carrier_month_csv.as_posix()}')
GROUP BY month
ORDER BY month;
''')

# Check: total flights in the file query should equal total flights in the DataFrame
print("Total flights from the CSV query:", month_from_file["total_flights"].sum())
print("Total flights from Pandas:       ", carrier_month_summary["flight_count"].sum())
month_from_file


### Example C: Join two DataFrames and check the key

Before a join, check that the key is unique in the lookup table. Compare `COUNT(*)` (all rows) with `COUNT(DISTINCT key)` (different key values). If the two numbers are equal, each key appears once, and the join will not duplicate rows.

In [ ]:
# A small lookup table: one row per airline
airlines_df = airlines.copy()

# Key check: is carrier unique in airlines_df?
key_check = run_duckdb('''
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT carrier) AS distinct_carriers
FROM airlines_df;
''')
display(key_check)

# Join two DataFrames in DuckDB
joined = run_duckdb('''
SELECT
    cm.carrier,
    cm.month,
    cm.flight_count,
    a.name AS airline_name
FROM carrier_month_summary AS cm
LEFT JOIN airlines_df AS a
    ON cm.carrier = a.carrier
ORDER BY cm.carrier, cm.month;
''')

print("Rows before the join:", len(carrier_month_summary), "| Rows after the join:", len(joined))
joined.head()


### Try it (optional)

Write one DuckDB query on `carrier_month_summary`. Use `GROUP BY`, and use `HAVING` or a join if you like. Then, in the second cell, compute the same result with Pandas and compare the two: do the row counts and values match?

This is the same check you will do in your project: one step in SQL, the same step in Pandas, and a comparison.

In [ ]:
# Try it - DuckDB query
# Use a DataFrame name as the table name, for example: FROM carrier_month_summary

query = '''

'''

my_duck_result = run_duckdb(query)
my_duck_result.head()


In [ ]:
# Try it - Pandas version of the same result
# Example pattern: carrier_month_summary.groupby("month").agg(...).reset_index()


**Path B checklist for your project notebook:**

1. Copy the `# COPY - Path B` cell into your project notebook. The Movies and Olist starter notebooks already have a "SQL Validation with DuckDB" section that you can build on.
2. Write at least one DuckDB query on your own DataFrames or CSV files that supports your integration or validation work.
3. Under each query, write one or two sentences on what the result tells you.

## 5. What a Good Project SQL Query Looks Like

With either path, a good query **supports your data work**. It is not a separate exercise. Strong examples:

- **Key check:** `COUNT(*)` vs `COUNT(DISTINCT key)` on your ABT, to show one row per unit of analysis.
- **Row count check:** row counts before and after a join.
- **Rebuild one step in SQL:** redo one aggregation of your ABT in SQL and compare the result with your Pandas result.
- **Lower-level table summary:** aggregate a detail table (for example, ratings per movie or items per order) before joining it.

For each query, show the SQL, its result, and one or two sentences on what the result tells you.